# Notebook 4: Model Evaluation & Comparison

Compare:
- Frequentist logistic regression (sklearn)
- Bayesian logistic regression (posterior predictive risk)
- Bayesian probit regression (posterior predictive risk)

Metrics and analyses:
- ROC-AUC, PR-AUC, Brier score, calibration
- WAIC/PSIS-LOO
- Bayesian Odds Ratio summary for logistic model
- Cost-sensitive Bayesian decision thresholding

In [ ]:
import numpy as np
import arviz as az
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
import pymc as pm
import sys; sys.path.append('..')

from src.preprocess import full_pipeline
from src.model import build_bayesian_logistic, build_bayesian_probit, posterior_predict
from src.evaluate import classification_metrics, plot_roc_curves, plot_calibration, plot_posterior_coefs

X_train, X_test, y_train, y_test, scaler, imputer, feature_names = full_pipeline('../data/framingham.csv')

# Load saved traces (from Notebook 03)
idata_logistic = az.from_netcdf('../data/idata_logistic.nc')
idata_probit   = az.from_netcdf('../data/idata_probit.nc')

In [ ]:
# Frequentist baseline
freq_model = LogisticRegression(max_iter=1000, random_state=42)
freq_model.fit(X_train, y_train)
y_prob_freq = freq_model.predict_proba(X_test)[:, 1]

print('=== Frequentist Logistic Regression ===')
classification_metrics(y_test, y_prob_freq)

In [ ]:
# Bayesian logistic: posterior predictive distribution (PPD)
logistic_model = build_bayesian_logistic(X_train, y_train)
ppc_logistic = posterior_predict(logistic_model, idata_logistic, X_test)

# Critical pitfall check:
# We must use continuous risk variable `p` (deterministic), NOT binary `y_obs`.
if 'p' not in ppc_logistic.posterior_predictive:
    raise RuntimeError('PPD missing `p`. Ensure sample_posterior_predictive(var_names=["p"]).')

# p_samples_log has shape (chain, draw, n_obs)
p_samples_log = np.asarray(ppc_logistic.posterior_predictive['p'])
# collapse chain/draw -> posterior samples per patient
p_samples_log = p_samples_log.reshape(-1, p_samples_log.shape[-1])

y_prob_bayes_logistic = p_samples_log.mean(axis=0)

hdi_log = az.hdi(p_samples_log.T, hdi_prob=0.95)  # (n_obs, 2)
p_hdi_low_log, p_hdi_high_log = hdi_log[:, 0], hdi_log[:, 1]

print('=== Bayesian Logistic Regression (PPD mean) ===')
classification_metrics(y_test, y_prob_bayes_logistic)

# Show prediction uncertainty for a few patients
idx = np.random.default_rng(42).choice(len(y_test), size=30, replace=False)
plt.figure(figsize=(10, 4))
plt.errorbar(
    x=np.arange(len(idx)),
    y=y_prob_bayes_logistic[idx],
    yerr=[y_prob_bayes_logistic[idx] - p_hdi_low_log[idx], p_hdi_high_log[idx] - y_prob_bayes_logistic[idx]],
    fmt='o', capsize=3,
)
plt.axhline(0.5, color='k', linestyle='--', alpha=0.3)
plt.title('Bayesian Logistic: individual risk with 95% HDI (subset)')
plt.xlabel('Random patient index (subset)')
plt.ylabel('Predicted risk')
plt.tight_layout()

In [ ]:
# Bayesian probit: posterior predictive distribution (PPD)
probit_model = build_bayesian_probit(X_train, y_train)
ppc_probit = posterior_predict(probit_model, idata_probit, X_test)

p_samples_prob = np.asarray(ppc_probit.posterior_predictive['p'])
p_samples_prob = p_samples_prob.reshape(-1, p_samples_prob.shape[-1])

y_prob_bayes_probit = p_samples_prob.mean(axis=0)

hdi_prob = az.hdi(p_samples_prob.T, hdi_prob=0.95)
p_hdi_low_prob, p_hdi_high_prob = hdi_prob[:, 0], hdi_prob[:, 1]

print('=== Bayesian Probit Regression (PPD mean) ===')
classification_metrics(y_test, y_prob_bayes_probit)

In [ ]:
# ROC / PR / Calibration comparison
from src.evaluate import plot_pr_curves

models = {
    'Frequentist LR': y_prob_freq,
    'Bayesian Logistic': y_prob_bayes_logistic,
    'Bayesian Probit': y_prob_bayes_probit,
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_roc_curves(models, y_test, ax=axes[0])
plot_pr_curves(models, y_test, ax=axes[1])
plot_calibration(models, y_test, ax=axes[2])
plt.tight_layout()

In [ ]:
# Posterior coefficient forest plot
plot_posterior_coefs(idata_logistic, feature_names=None, model_name='Bayesian Logistic')

# Bayesian Odds Ratios (OR) for clinical interpretability (logistic model only)
# Important: OR = exp(beta) is valid for logistic link, NOT for probit.
import pandas as pd
from src.preprocess import load_data
from src.evaluate import logistic_odds_ratio_summary

feature_names = load_data('../data/framingham.csv').drop(columns=['TenYearCHD']).columns.tolist()
or_summary = logistic_odds_ratio_summary(idata_logistic, feature_names, hdi_prob=0.95)

display(or_summary.head(10)[['feature', 'or_median', 'or_hdi_95_low', 'or_hdi_95_high']])

# Probit coefficients can be compared on latent-scale (optional approximation):
# beta_logistic ≈ 1.6 * beta_probit

In [ ]:
# WAIC / LOO model comparison (requires log_likelihood)
# If log_likelihood is missing, rerun Notebook 03 (sample_model computes it).
try:
    cmp_waic = az.compare({'logistic': idata_logistic, 'probit': idata_probit}, ic='waic')
    cmp_loo = az.compare({'logistic': idata_logistic, 'probit': idata_probit}, ic='loo')
    display(cmp_waic)
    display(cmp_loo)
except Exception as e:
    print('Model comparison failed:', e)

# Bayesian decision analysis under asymmetric costs
# Example: false negative (missed CHD) costs 5x false positive (unnecessary follow-up)
C_FP = 1.0
C_FN = 5.0

# For probabilistic classifiers, Bayes-optimal threshold under this loss is:
# Predict positive if C_FP*(1-p) < C_FN*p  ->  p > C_FP/(C_FP + C_FN)
t_star = C_FP / (C_FP + C_FN)
print('Bayes-optimal threshold (cost-based):', round(t_star, 3))

# Compute expected loss across thresholds using posterior mean probabilities
thresholds = np.linspace(0.01, 0.99, 99)
losses = []
for t in thresholds:
    y_hat = (y_prob_bayes_logistic >= t).astype(int)
    fp = np.sum((y_hat == 1) & (y_test == 0))
    fn = np.sum((y_hat == 0) & (y_test == 1))
    losses.append(C_FP * fp + C_FN * fn)
losses = np.array(losses)

best_idx = int(np.argmin(losses))
t_emp = thresholds[best_idx]
print('Empirical best threshold (min observed cost):', round(float(t_emp), 3))

plt.figure(figsize=(7, 4))
plt.plot(thresholds, losses, label='Observed cost on test set')
plt.axvline(t_star, color='tomato', linestyle='--', label=f'Bayes threshold={t_star:.3f}')
plt.axvline(t_emp, color='steelblue', linestyle='--', label=f'Empirical best={t_emp:.3f}')
plt.xlabel('Decision threshold')
plt.ylabel('Cost (C_FP*FP + C_FN*FN)')
plt.title('Decision analysis with asymmetric costs')
plt.legend()
plt.tight_layout()